# 과제 1. Data Processing with Pandas
2026-2 인공지능의 이해 / Santander Product Recommendation `train_ver2.csv`

- Q1. 마지막 달(2016-05-28) 도시별 고객 수, 평균 소득, 소득 중앙값, 평균 나이를 구하고 고객 50명 이상 도시 중 평균 소득 상위 5개 출력
- Q2. 전월 대비 0 → 1이 된 상품을 월별·상품별로 세고, 전체 기간 신규 가입 상위 5개 상품 출력
- Q3. 마지막 달 가입 상품 수 3개 이상 여부와 가장 관련 있는 기준을 chi-square로 찾기

## 0. 데이터 불러오기

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

DATA_PATH = 'train_ver2.csv'   # Colab + Google Drive라면 'drive/MyDrive/train_ver2.csv'
LAST_MONTH = '2016-05-28'

In [2]:
# 상품 컬럼은 25~48번째 열 (인덱스 24~47)
header = pd.read_csv(DATA_PATH, nrows=0).columns
product_cols = list(header[24:48])
print(len(product_cols), product_cols)

24 ['ind_ahor_fin_ult1', 'ind_aval_fin_ult1', 'ind_cco_fin_ult1', 'ind_cder_fin_ult1', 'ind_cno_fin_ult1', 'ind_ctju_fin_ult1', 'ind_ctma_fin_ult1', 'ind_ctop_fin_ult1', 'ind_ctpp_fin_ult1', 'ind_deco_fin_ult1', 'ind_deme_fin_ult1', 'ind_dela_fin_ult1', 'ind_ecue_fin_ult1', 'ind_fond_fin_ult1', 'ind_hip_fin_ult1', 'ind_plan_fin_ult1', 'ind_pres_fin_ult1', 'ind_reca_fin_ult1', 'ind_tjcr_fin_ult1', 'ind_valo_fin_ult1', 'ind_viv_fin_ult1', 'ind_nomina_ult1', 'ind_nom_pens_ult1', 'ind_recibo_ult1']


파일이 약 2GB(1,360만 행)라서 100만 행씩 나눠 읽었다.
- `df`: 전체 기간의 날짜, 고객 ID, 상품 24개 (Q2용)
- `last`: 마지막 달 행만, 고객 정보 포함 (Q1, Q3용)

상품 값이 비어 있는 경우는 0으로 채우지 않고 -1로 두었다. 0으로 채우면 '모름 → 1'이 신규 가입으로 잘못 세질 수 있기 때문이다. 메모리를 줄이려고 상품 컬럼은 int8로 바꿨다.

In [3]:
info_cols = ['ind_empleado', 'sexo', 'age', 'nomprov', 'renta', 'segmento', 'tiprel_1mes']
dtypes = {'ncodpers': 'int32', 'fecha_dato': 'str', 'renta': 'float64'}
dtypes.update({c: 'str' for c in info_cols if c != 'renta'})
dtypes.update({c: 'float32' for c in product_cols})

df_parts, last_parts = [], []
for chunk in pd.read_csv(DATA_PATH, usecols=['fecha_dato', 'ncodpers'] + info_cols + product_cols,
                         dtype=dtypes, chunksize=1_000_000, low_memory=False):
    chunk[product_cols] = chunk[product_cols].fillna(-1).astype('int8')
    last_parts.append(chunk.loc[chunk['fecha_dato'] == LAST_MONTH])
    chunk['fecha_dato'] = pd.to_datetime(chunk['fecha_dato'])
    df_parts.append(chunk[['fecha_dato', 'ncodpers'] + product_cols])

df = pd.concat(df_parts, ignore_index=True)
last = pd.concat(last_parts, ignore_index=True)
del df_parts, last_parts

print('df:', df.shape)
print('last:', last.shape)

df: (13647309, 26)
last: (931453, 33)


In [4]:
months = sorted(df['fecha_dato'].unique())
print('기간:', months[0].date(), '~', months[-1].date(), f'({len(months)}개월)')
print('전체 고객 수:', df['ncodpers'].nunique())

missing = (df[product_cols] == -1).sum()
print('\n상품 값이 비어 있는 행 수')
print(missing.loc[missing > 0])

기간: 2015-01-28 ~ 2016-05-28 (17개월)
전체 고객 수: 956645



상품 값이 비어 있는 행 수
ind_nomina_ult1      16063
ind_nom_pens_ult1    16063
dtype: int64


In [5]:
# age는 ' 35', ' NA' 같은 문자열이라 공백을 지우고 숫자로 변환 (변환 안 되면 NaN)
last['age'] = pd.to_numeric(last['age'].str.strip(), errors='coerce')

# 한 달 안에서 고객 1명 = 1행인지 확인 (행 수를 고객 수로 써도 되는지)
print('중복 고객 ID:', last['ncodpers'].duplicated().sum())
print('마지막 달 고객 수:', len(last))
print('결측 - age:', last['age'].isna().sum(), '/ renta:', last['renta'].isna().sum(),
      '/ nomprov:', last['nomprov'].isna().sum())

중복 고객 ID: 0
마지막 달 고객 수: 931453
결측 - age: 0 / renta: 229018 / nomprov: 3992


## Q1. 마지막 달 도시별 통계
도시(`nomprov`)별로 고객 수, 평균 소득, 소득 중앙값, 평균 나이를 각각 `groupby`로 구한 뒤 `join`으로 합쳤다. 그다음 고객 수 50명 이상인 도시만 `.loc`로 남기고 평균 소득 순으로 정렬했다.

소득이나 나이가 비어 있는 고객은 평균·중앙값 계산에서만 빠지고 고객 수에는 들어간다. 도시가 비어 있는 고객은 `groupby`에서 빠진다. 마지막 달에는 모든 도시가 고객 50명 이상이라 이 조건으로 빠지는 도시는 없다.

In [6]:
n_customers = last.groupby(['nomprov'], as_index=False)['ncodpers'].count() \
                  .rename(columns={'ncodpers': 'n_customers'})
mean_income = last.groupby(['nomprov'], as_index=False)['renta'].mean() \
                  .rename(columns={'renta': 'mean_income'})
median_income = last.groupby(['nomprov'], as_index=False)['renta'].median() \
                    .rename(columns={'renta': 'median_income'})
mean_age = last.groupby(['nomprov'], as_index=False)['age'].mean() \
               .rename(columns={'age': 'mean_age'})

city_stats = n_customers.join(mean_income.set_index('nomprov'), on='nomprov') \
                        .join(median_income.set_index('nomprov'), on='nomprov') \
                        .join(mean_age.set_index('nomprov'), on='nomprov')

print('도시 수:', len(city_stats), '/ 가장 적은 고객 수:', city_stats['n_customers'].min())
city_stats.sort_values('n_customers', ascending=False).head(10)

도시 수: 52 / 가장 적은 고객 수: 513


,nomprov,n_customers,mean_income,median_income,mean_age
30,MADRID,298841,"178,362.99","138,024.14",45.21
8,BARCELONA,88754,"164,584.84","130,877.16",39.87
48,VALENCIA,48103,"89,296.19","72,734.91",39.11
43,SEVILLA,40581,"117,368.87","92,369.46",38.16
18,"CORUÑA, A",28771,"112,581.08","97,389.32",33.80
33,MURCIA,27795,"78,943.24","67,794.74",31.88
31,MALAGA,24583,"120,920.86","94,961.91",38.50
51,ZARAGOZA,23198,"110,172.52","99,607.89",41.01
2,ALICANTE,22210,"87,142.96","67,332.87",38.88
12,CADIZ,19829,"98,772.35","79,068.72",35.83


In [7]:
q1 = city_stats.loc[city_stats['n_customers'] >= 50] \
               .sort_values('mean_income', ascending=False) \
               .head(5) \
               .reset_index(drop=True)
q1

,nomprov,n_customers,mean_income,median_income,mean_age
0,CEUTA,513,"200,515.89","128,699.58",44.01
1,MADRID,298841,"178,362.99","138,024.14",45.21
2,"BALEARS, ILLES",9143,"171,890.95","122,837.22",39.75
3,BARCELONA,88754,"164,584.84","130,877.16",39.87
4,MELILLA,641,"149,071.94","117,718.37",45.33


## Q2. 전월 대비 신규 가입 상품 (0 → 1)
전월에 0이고 이번 달에 1이면 그 달에 새로 가입한 것으로 본다.

1. `.loc`로 이번 달과 전월 데이터를 고른다.
2. 고객 ID로 `join(how='inner')`해서 두 달 모두 기록이 있는 고객만 남긴다.
3. 상품마다 `(전월 == 0) & (이번 달 == 1)`을 계산하고 `.sum()`으로 True 개수를 센다.
4. 2015-02부터 2016-05까지 16번 반복해서 월별·상품별 표를 만든다.

첫 달(2015-01)은 전월이 없어서 제외된다. 값이 -1(결측)이면 0도 1도 아니라서 세지 않는다.

In [8]:
# 달이 하나도 빠지지 않고 이어지는지 확인 (그래야 바로 앞 달 = 전월)
month_no = [m.year * 12 + m.month for m in months]
assert all(b - a == 1 for a, b in zip(month_no[:-1], month_no[1:]))

prev_cols = [c + '_prev' for c in product_cols]
monthly_rows = []

for i in range(1, len(months)):
    cur = df.loc[df['fecha_dato'] == months[i], ['ncodpers'] + product_cols]
    prev = df.loc[df['fecha_dato'] == months[i - 1], ['ncodpers'] + product_cols]

    both = cur.join(prev.set_index('ncodpers'), on='ncodpers', how='inner', rsuffix='_prev')

    before = both[prev_cols]
    before.columns = product_cols                      # 컬럼 이름을 맞춰야 같은 상품끼리 비교됨
    new = (before == 0) & (both[product_cols] == 1)

    counts = new.sum()
    counts.name = months[i].strftime('%Y-%m-%d')
    monthly_rows.append(counts)

monthly_new = pd.DataFrame(monthly_rows)               # 행: 월 16개, 열: 상품 24개
monthly_new.index.name = 'fecha_dato'
monthly_new

,ind_ahor_fin_ult1,ind_aval_fin_ult1,ind_cco_fin_ult1,ind_cder_fin_ult1,ind_cno_fin_ult1,ind_ctju_fin_ult1,ind_ctma_fin_ult1,ind_ctop_fin_ult1,ind_ctpp_fin_ult1,ind_deco_fin_ult1,...,ind_hip_fin_ult1,ind_plan_fin_ult1,ind_pres_fin_ult1,ind_reca_fin_ult1,ind_tjcr_fin_ult1,ind_valo_fin_ult1,ind_viv_fin_ult1,ind_nomina_ult1,ind_nom_pens_ult1,ind_recibo_ult1
fecha_dato,,,,,,,,,,,,,,,,,,,,,
2015-02-28,1,1,2132,12,1884,7,291,249,171,348,...,5,27,9,217,3471,259,8,5659,5656,8285
2015-03-28,0,1,2293,15,1786,18,272,284,165,481,...,7,24,12,217,5076,205,4,4479,4473,9135
2015-04-28,0,1,2366,12,1984,10,217,278,167,263,...,4,22,17,871,4627,194,5,2663,5628,8346
2015-05-28,0,0,2264,7,1900,10,205,289,172,238,...,9,21,11,343,4071,201,4,3369,3345,7085
2015-06-28,0,0,6588,9,1857,7,210,213,151,289,...,4,19,8,2931,4655,152,3,5070,8135,9023
2015-07-28,0,0,3056,10,2771,18,191,252,164,265,...,6,27,5,589,4886,232,9,4408,4850,9172
2015-08-28,0,0,5730,10,2412,37,240,231,145,322,...,1,22,13,264,4379,445,2,2950,2968,7115
2015-09-28,0,0,4389,7,1775,22,471,191,122,387,...,8,21,13,271,4463,328,2,4683,4753,11210
2015-10-28,0,0,5682,3,2409,43,468,223,153,349,...,7,19,9,593,4461,911,2,4818,4790,12100


In [9]:
total_new = monthly_new.sum().sort_values(ascending=False)
q2 = total_new.head(5).reset_index()
q2.columns = ['product', 'new_count']
q2

,product,new_count
0,ind_recibo_ult1,153146
1,ind_nom_pens_ult1,84709
2,ind_nomina_ult1,73753
3,ind_tjcr_fin_ult1,69118
4,ind_cco_fin_ult1,69103


## Q3. 가입 상품 3개 이상 여부와 가장 관련 있는 기준
수업(Ch03)의 chi-square 계산을 함수로 만들었다.
- 기준 × 목표(3개 이상/미만) 교차표에서 observed(실제 고객 수)를 구한다.
- expected = 행 합계 × 열 합계 / 전체
- χ² = Σ (observed − expected)² / expected 이고, 값이 클수록 관련이 크다.

함수가 맞는지 먼저 수업 예제(Breakfast × Test score, 1.742)로 확인했다.

과제의 기준과 데이터 열은 다음과 같이 대응된다. 1) 성별 → `sexo`, 2) 나이 구간 → `age`로 만든 `age_group`, 3) segment → `segmento`, 4) `ind_empleado`, 5) `tiprel_1mes`

In [10]:
def chi_square(data, col, target):
    d = data[[col, target]].dropna()              # 값이 없는 고객은 제외
    observed = pd.crosstab(d[col], d[target])
    row_sum = observed.sum(axis=1)
    col_sum = observed.sum(axis=0)
    n = observed.to_numpy().sum()
    expected = pd.DataFrame(np.outer(row_sum, col_sum) / n,
                            index=observed.index, columns=observed.columns)
    chi2_value = ((observed - expected) ** 2 / expected).to_numpy().sum()
    return chi2_value, observed, expected

# 수업 예제로 확인
toy = pd.DataFrame({'Breakfast': ['Y', 'N', 'Y', 'Y', 'N', 'N', 'Y', 'Y'],
                    'Test':      ['P', 'F', 'F', 'P', 'P', 'F', 'P', 'P']})
toy_chi2, toy_obs, toy_exp = chi_square(toy, 'Breakfast', 'Test')
print(toy_obs, '\n')
print(toy_exp.round(3), '\n')
print(f'chi2 = {toy_chi2:.3f}')

Test       F  P
Breakfast      
N          2  1
Y          1  4 

Test         F    P
Breakfast          
N         1.12 1.88
Y         1.88 3.12 

chi2 = 1.742


In [11]:
# 목표: 가입 상품(값이 1인 상품) 개수가 3개 이상인지
last['n_products'] = (last[product_cols] == 1).sum(axis=1)
last['target'] = last['n_products'].apply(lambda n: '3개 이상' if n >= 3 else '3개 미만')

# 나이 구간: 65 이상 senior, 30 이상 65 미만 adult, 30 미만 young
def age_group(age):
    if pd.isna(age):
        return None
    if age >= 65:
        return 'senior'
    if age >= 30:
        return 'adult'
    return 'young'

last['age_group'] = last['age'].apply(age_group)

print(last['target'].value_counts(), '\n')
print(last['age_group'].value_counts())

target
3개 미만    808432
3개 이상    123021
Name: count, dtype: int64 

age_group
adult     490900
young     348610
senior     91943
Name: count, dtype: int64


In [12]:
criteria = ['sexo', 'age_group', 'segmento', 'ind_empleado', 'tiprel_1mes']

rows, tables = [], {}
for col in criteria:
    chi2_value, observed, expected = chi_square(last, col, 'target')
    n = observed.to_numpy().sum()
    dof = (observed.shape[0] - 1) * (observed.shape[1] - 1)
    rows.append({'criterion': col,
                 'chi2': chi2_value,
                 'dof': dof,
                 'p_value': stats.chi2.sf(chi2_value, dof),
                 'cramers_v': np.sqrt(chi2_value / (n * (min(observed.shape) - 1))),
                 'min_expected': expected.to_numpy().min()})
    tables[col] = (observed, expected)

q3 = pd.DataFrame(rows).sort_values('chi2', ascending=False).reset_index(drop=True)
q3.index = q3.index + 1
q3

,criterion,chi2,dof,p_value,cramers_v,min_expected
1,tiprel_1mes,"183,808.59",3,0.00,0.45,10.21
2,segmento,"124,374.55",2,0.00,0.37,"4,711.77"
3,age_group,"61,002.20",2,0.00,0.26,"12,143.31"
4,sexo,"3,412.59",1,0.00,0.06,"56,246.25"
5,ind_empleado,"1,343.69",4,0.00,0.04,0.13


In [13]:
# 기준 값별로 '3개 이상' 고객 비율 확인 (결과 해석용)
for col in criteria:
    observed, expected = tables[col]
    view = observed.copy()
    view['3개 이상 비율'] = observed['3개 이상'] / observed.sum(axis=1)
    print(f'[{col}]')
    print(view.round(3), '\n')

[sexo]
target   3개 미만  3개 이상  3개 이상 비율
sexo                           
H       379129  46737      0.11
V       429298  76284      0.15 

[age_group]
target      3개 미만   3개 이상  3개 이상 비율
age_group                          
adult      388734  102166      0.21
senior      79101   12842      0.14
young      340597    8013      0.02 

[segmento]
target               3개 미만  3개 이상  3개 이상 비율
segmento                                   
01 - TOP             11658  23757      0.67
02 - PARTICULARES   454077  88519      0.16
03 - UNIVERSITARIO  335703  10714      0.03 

[ind_empleado]
target         3개 미만   3개 이상  3개 이상 비율
ind_empleado                          
A                 18     130      0.88
B                104     115      0.53
F                 57      95      0.62
N             808253  122680      0.13
S                  0       1      1.00 

[tiprel_1mes]
target        3개 미만   3개 이상  3개 이상 비율
tiprel_1mes                          
A            271352  121255      0.31
I            53269

In [14]:
# 직접 계산한 값이 scipy 결과와 같은지 확인 (수업 공식과 맞추려고 correction=False)
for col in criteria:
    observed, _ = tables[col]
    sp_chi2 = stats.chi2_contingency(observed, correction=False)[0]
    my_chi2 = q3.loc[q3['criterion'] == col, 'chi2'].iloc[0]
    print(col, round(my_chi2, 2), round(sp_chi2, 2), np.isclose(my_chi2, sp_chi2))

sexo 3412.59 3412.59 True
age_group 61002.2 61002.2 True
segmento 124374.55 124374.55 True
ind_empleado 1343.69 1343.69 True
tiprel_1mes 183808.59 183808.59 True


## 결론
- Q1: 고객 50명 이상 도시 중 평균 소득 상위 5개는 CEUTA, MADRID, `BALEARS, ILLES`, BARCELONA, MELILLA 순이다.
- Q2: 전체 기간 신규 가입이 가장 많은 상품 5개는 ind_recibo_ult1, ind_nom_pens_ult1, ind_nomina_ult1, ind_tjcr_fin_ult1, ind_cco_fin_ult1 순이다.
- Q3: χ²가 가장 큰 기준은 tiprel_1mes이다. 그다음은 segmento, age_group, sexo, ind_empleado 순이다. 범주 수 차이를 보정한 Cramér's V로 봐도 순위가 같다.

참고
- 고객이 약 93만 명이라 p-value는 모두 0에 가깝다. 그래서 순위는 χ² 값으로 비교했다.
- ind_empleado는 과제 설명에 없는 S 값이 1명 있어서 expected가 5보다 작은 칸이 있다(min_expected 0.13). 이 기준의 χ²는 조심해서 봐야 하지만 순위는 마지막이라 결론에는 영향이 없다.
- chi-square는 관련성만 보여 주고 원인과 결과를 뜻하지는 않는다.